# E-Commerce Order Analytics System (Colab version)

An end-to-end mini project



**Steps:**
1. Install libraries
2. Generate messy raw data (Faker)
3. Clean data with pandas
4. Load into a SQLite database (with constraints)
5. SQL analytics: joins & aggregations
6. SQL analytics: window functions & CTEs
7. Cohort, retention & RFM segmentation
8. Simple interactive reporting tool
9. Edge case tests


## 1. Install & import libraries

In [23]:
!pip install -q faker tabulate


In [24]:
import os
import random
import sqlite3
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker
from tabulate import tabulate

random.seed(42)
fake = Faker()
Faker.seed(42)

# Folder layout (created fresh each run, mirrors the GitHub repo structure)
BASE_DIR = "ecommerce-analytics-system"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
CLEAN_DIR = os.path.join(BASE_DIR, "data", "cleaned")
SQL_DIR = os.path.join(BASE_DIR, "sql")
DB_PATH = os.path.join(BASE_DIR, "ecommerce.db")

for d in [RAW_DIR, CLEAN_DIR, SQL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Folders ready:", RAW_DIR, CLEAN_DIR, SQL_DIR)


Folders ready: ecommerce-analytics-system/data/raw ecommerce-analytics-system/data/cleaned ecommerce-analytics-system/sql


## 2. Step 1 — Generate realistic (messy) datasets

We create `customers`, `products`, `orders`, `order_items` with **intentional
issues** so we have something real to clean in the next step:

- duplicate rows
- missing values (null email, null city, null category)
- orphan foreign keys (an order pointing to a customer_id that doesn't exist)
- invalid dates (future dates, blank dates)
- invalid numbers (negative price, negative/missing quantity)


In [25]:
CATEGORIES = [
    "Electronics", "Clothing", "Home & Kitchen", "Books",
    "Sports", "Toys", "Beauty", "Grocery", None  # None -> intentional missing category
]
CITIES = [
    "Mumbai", "Delhi", "Bengaluru", "Jaipur", "Pune",
    "Hyderabad", "Chennai", "Kolkata", None  # None -> missing city
]
ORDER_STATUSES = ["completed", "pending", "cancelled", "returned"]

N_CUSTOMERS = 200
N_PRODUCTS = 60
N_ORDERS = 900


def generate_customers(n):
    rows = []
    for cid in range(1, n + 1):
        first, last = fake.first_name(), fake.last_name()
        email = f"{first.lower()}.{last.lower()}{random.randint(1, 999)}@example.com"
        rows.append({
            "customer_id": cid, "first_name": first, "last_name": last,
            "email": email,
            "signup_date": fake.date_between(start_date="-3y", end_date="today"),
            "city": random.choice(CITIES),
        })
    df = pd.DataFrame(rows)
    df = pd.concat([df, df.sample(5, random_state=1)], ignore_index=True)  # duplicates
    df.loc[df.sample(4, random_state=2).index, "email"] = None            # missing emails
    return df


def generate_products(n):
    rows = []
    for pid in range(1, n + 1):
        rows.append({
            "product_id": pid,
            "product_name": fake.word().capitalize() + " " + fake.word().capitalize(),
            "category": random.choice(CATEGORIES),
            "price": round(random.uniform(5, 500), 2),
        })
    df = pd.DataFrame(rows)
    bad_idx = df.sample(3, random_state=3).index
    df.loc[bad_idx, "price"] = [-10.0, 0, -5.5][: len(bad_idx)]           # bad prices
    df = pd.concat([df, df.sample(2, random_state=4)], ignore_index=True)  # duplicates
    return df


def generate_orders(n, customer_ids):
    rows = []
    today = datetime.today().date()
    for oid in range(1, n + 1):
        cust_id = (max(customer_ids) + random.randint(1, 50)) if random.random() < 0.03 else random.choice(customer_ids)
        order_date = fake.date_between(start_date="-2y", end_date="today")
        if random.random() < 0.02:
            order_date = today + timedelta(days=random.randint(1, 100))    # future date
        if random.random() < 0.01:
            order_date = None                                              # missing date
        rows.append({
            "order_id": oid, "customer_id": cust_id,
            "order_date": order_date, "status": random.choice(ORDER_STATUSES),
        })
    df = pd.DataFrame(rows)
    df = pd.concat([df, df.sample(6, random_state=5)], ignore_index=True)  # duplicates
    return df


def generate_order_items(order_ids, product_ids):
    rows, item_id = [], 1
    for oid in order_ids:
        for _ in range(random.randint(1, 4)):
            pid = (max(product_ids) + random.randint(1, 20)) if random.random() < 0.02 else random.choice(product_ids)
            qty = random.randint(1, 5)
            if random.random() < 0.02:
                qty = None
            elif random.random() < 0.01:
                qty = -2
            rows.append({
                "order_item_id": item_id, "order_id": oid, "product_id": pid,
                "quantity": qty, "unit_price": round(random.uniform(5, 500), 2),
            })
            item_id += 1
    for _ in range(10):  # orphan order_items pointing to a non-existent order
        rows.append({
            "order_item_id": item_id,
            "order_id": max(order_ids) + random.randint(1, 30),
            "product_id": random.choice(product_ids),
            "quantity": random.randint(1, 3),
            "unit_price": round(random.uniform(5, 500), 2),
        })
        item_id += 1
    df = pd.DataFrame(rows)
    df = pd.concat([df, df.sample(8, random_state=6)], ignore_index=True)  # duplicates
    return df


customers = generate_customers(N_CUSTOMERS)
products = generate_products(N_PRODUCTS)
orders = generate_orders(N_ORDERS, customers["customer_id"].unique().tolist())
order_items = generate_order_items(orders["order_id"].unique().tolist(), products["product_id"].unique().tolist())

customers.to_csv(os.path.join(RAW_DIR, "customers.csv"), index=False)
products.to_csv(os.path.join(RAW_DIR, "products.csv"), index=False)
orders.to_csv(os.path.join(RAW_DIR, "orders.csv"), index=False)
order_items.to_csv(os.path.join(RAW_DIR, "order_items.csv"), index=False)

print(f"customers: {len(customers)} rows | products: {len(products)} rows "
      f"| orders: {len(orders)} rows | order_items: {len(order_items)} rows")
customers.head()


customers: 205 rows | products: 62 rows | orders: 906 rows | order_items: 2227 rows


,customer_id,first_name,last_name,email,signup_date,city
0,1,Danielle,Johnson,danielle.johnson655@example.com,2024-05-29,Delhi
1,2,Jeffrey,Doyle,jeffrey.doyle26@example.com,2025-08-12,Pune
2,3,Patricia,Miller,patricia.miller251@example.com,2024-11-06,Jaipur
3,4,Anthony,Robinson,anthony.robinson143@example.com,2025-02-05,Delhi
4,5,Anthony,Gonzalez,anthony.gonzalez693@example.com,2025-07-14,None


## 3. Step 2 — Clean the data with pandas

For each table: drop duplicates, fix/fill missing values, fix data types,
and check **referential integrity** (drop rows whose foreign key doesn't
exist in the parent table). We print how many rows were dropped and why,
so nothing silently disappears.


In [46]:
def clean_customers(df):
    before = len(df)
    df = df.drop_duplicates().drop_duplicates(subset="customer_id", keep="first")
    df["city"] = df["city"].fillna("Unknown")
    df["email"] = df["email"].fillna("unknown@example.com")
    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
    print(f"customers: {before} -> {len(df)} rows")
    return df


def clean_products(df):
    before = len(df)
    df = df.drop_duplicates().drop_duplicates(subset="product_id", keep="first")
    df["category"] = df["category"].fillna("Uncategorized")
    median_price = df.loc[df["price"] > 0, "price"].median()
    df.loc[df["price"] <= 0, "price"] = median_price
    print(f"products: {before} -> {len(df)} rows")
    return df


def clean_orders(df, valid_customer_ids):
    before = len(df)
    df = df.drop_duplicates().drop_duplicates(subset="order_id", keep="first")
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    missing_dates = df["order_date"].isna().sum()
    df = df.dropna(subset=["order_date"])

    today = pd.Timestamp(datetime.today().date())
    future_dates = (df["order_date"] > today).sum()
    df = df[df["order_date"] <= today]

    orphans = (~df["customer_id"].isin(valid_customer_ids)).sum()
    df = df[df["customer_id"].isin(valid_customer_ids)]

    print(f"orders: {before} -> {len(df)} rows "
          f"(dropped {missing_dates} missing dates, {future_dates} future dates, {orphans} orphan customer_id)")
    return df


def clean_order_items(df, valid_order_ids, valid_product_ids):
    before = len(df)
    df = df.drop_duplicates().drop_duplicates(subset="order_item_id", keep="first")

    orphan_orders = (~df["order_id"].isin(valid_order_ids)).sum()
    orphan_products = (~df["product_id"].isin(valid_product_ids)).sum()
    df = df[df["order_id"].isin(valid_order_ids)]
    df = df[df["product_id"].isin(valid_product_ids)]

    df["quantity"] = df["quantity"].fillna(1)
    invalid_qty = (df["quantity"] <= 0).sum()
    df = df[df["quantity"] > 0]
    df["quantity"] = df["quantity"].astype(int)

    print(f"order_items: {before} -> {len(df)} rows "
          f"(dropped {orphan_orders} orphan order_id, {orphan_products} orphan product_id, {invalid_qty} invalid quantity)")
    return df


customers_clean = clean_customers(customers)
products_clean = clean_products(products)
orders_clean = clean_orders(orders, set(customers_clean["customer_id"]))
order_items_clean = clean_order_items(order_items, set(orders_clean["order_id"]), set(products_clean["product_id"]))

customers_clean.to_csv(os.path.join(CLEAN_DIR, "customers_clean.csv"), index=False)
products_clean.to_csv(os.path.join(CLEAN_DIR, "products_clean.csv"), index=False)
orders_clean.to_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), index=False)
order_items_clean.to_csv(os.path.join(CLEAN_DIR, "order_items_clean.csv"), index=False)


customers: 205 -> 200 rows
products: 62 -> 60 rows
orders: 906 -> 856 rows (dropped 8 missing dates, 15 future dates, 21 orphan customer_id)
order_items: 2227 -> 2052 rows (dropped 109 orphan order_id, 42 orphan product_id, 21 invalid quantity)


## 4. Step 3 — Load into a SQLite database (with constraints)

We build the schema with `PRIMARY KEY` / `FOREIGN KEY` / `NOT NULL` /
`CHECK` constraints, then load the cleaned data in and verify counts.


In [27]:
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    first_name    TEXT NOT NULL,
    last_name     TEXT NOT NULL,
    email         TEXT NOT NULL,
    signup_date   DATE,
    city          TEXT
);

CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT NOT NULL,
    category      TEXT,
    price         REAL NOT NULL CHECK (price > 0)
);

CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL,
    order_date    DATE NOT NULL,
    status        TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    unit_price    REAL NOT NULL CHECK (unit_price > 0),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);
"""

with open(os.path.join(SQL_DIR, "schema.sql"), "w") as f:
    f.write(SCHEMA_SQL)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(SCHEMA_SQL)

customers_clean.to_sql("customers", conn, if_exists="append", index=False)
products_clean.to_sql("products", conn, if_exists="append", index=False)
orders_clean.to_sql("orders", conn, if_exists="append", index=False)
order_items_clean.to_sql("order_items", conn, if_exists="append", index=False)
conn.commit()

print("Row counts in database:")
for table in ["customers", "products", "orders", "order_items"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count}")

orphan_items = conn.execute("""
    SELECT COUNT(*) FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""").fetchone()[0]
print(f"  order_items with no matching order (should be 0): {orphan_items}")


Row counts in database:
  customers: 200
  products: 60
  orders: 856
  order_items: 2052
  order_items with no matching order (should be 0): 0


## 5. Step 4 — SQL Analytics: Joins & Aggregations

Total revenue per customer / category / month, top products, and average
order value (AOV) by customer segment.


In [28]:
pd.read_sql("""
    SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_name,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, customer_name
    ORDER BY total_revenue DESC
    LIMIT 10;
""", conn)


,customer_id,customer_name,total_revenue
0,68,Sara Allison,21137.19
1,64,Shane Henderson,20066.90
2,5,Anthony Gonzalez,19595.48
3,197,Nicholas Johnson,19448.03
4,165,Ryan Johnson,18969.10
5,153,Katie Boyd,18274.81
6,155,Joseph Flores,18184.57
7,138,Karen Chambers,17774.91
8,175,Debra Harrington,17619.28
9,191,Jasmine Graham,17598.02


In [29]:
pd.read_sql("""
    SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC;
""", conn)


,category,total_revenue
0,Books,295313.18
1,Sports,288863.89
2,Electronics,205649.03
3,Clothing,193059.81
4,Beauty,172008.09
5,Home & Kitchen,169120.69
6,Uncategorized,128104.10
7,Toys,103721.67
8,Grocery,40316.39


In [30]:
pd.read_sql("""
    SELECT strftime('%Y-%m', o.order_date) AS order_month,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY order_month
    ORDER BY order_month;
""", conn)


,order_month,total_revenue
0,2024-08,60575.47
1,2024-09,65129.00
2,2024-10,58908.19
3,2024-11,77271.76
4,2024-12,50434.44
5,2025-01,56903.78
6,2025-02,66309.47
7,2025-03,59981.12
8,2025-04,66681.99
9,2025-05,56177.97


In [31]:
pd.read_sql("""
    SELECT p.product_id, p.product_name,
           SUM(oi.quantity) AS total_quantity_sold,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.product_name
    ORDER BY total_revenue DESC
    LIMIT 10;
""", conn)


,product_id,product_name,total_quantity_sold,total_revenue
0,45,Actually Painting,164,42350.03
1,44,Board Turn,134,41812.48
2,2,Service Investment,153,41327.20
3,17,Oil Give,125,36366.18
4,10,Do Interview,128,36343.64
5,28,Member Factor,123,35775.11
6,56,Argue Civil,116,32609.85
7,6,Race Seek,118,32434.57
8,7,Military Line,132,32202.36
9,46,Feel Manager,110,32159.30


In [32]:
pd.read_sql("""
    WITH order_counts AS (
        SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id
    ),
    segments AS (
        SELECT customer_id,
               CASE WHEN n_orders = 1 THEN 'one_time'
                    WHEN n_orders BETWEEN 2 AND 4 THEN 'occasional'
                    ELSE 'loyal' END AS segment
        FROM order_counts
    ),
    order_values AS (
        SELECT o.order_id, o.customer_id, SUM(oi.quantity * oi.unit_price) AS order_value
        FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY o.order_id, o.customer_id
    )
    SELECT s.segment, ROUND(AVG(ov.order_value), 2) AS avg_order_value,
           COUNT(DISTINCT ov.order_id) AS num_orders
    FROM order_values ov JOIN segments s ON s.customer_id = ov.customer_id
    GROUP BY s.segment
    ORDER BY avg_order_value DESC;
""", conn)


,segment,avg_order_value,num_orders
0,one_time,2054.77,10
1,loyal,1899.57,520
2,occasional,1836.97,320


## 6. Step 5 — Window Functions & CTEs

`RANK()`/`DENSE_RANK()` for customer lifetime value, running totals and
moving averages of monthly revenue, and month-over-month growth rate.


In [33]:
pd.read_sql("""
    WITH customer_ltv AS (
        SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_name,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS lifetime_value
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY c.customer_id, customer_name
    )
    SELECT customer_id, customer_name, lifetime_value,
           RANK() OVER (ORDER BY lifetime_value DESC) AS ltv_rank,
           DENSE_RANK() OVER (ORDER BY lifetime_value DESC) AS ltv_dense_rank
    FROM customer_ltv
    ORDER BY lifetime_value DESC
    LIMIT 10;
""", conn)


,customer_id,customer_name,lifetime_value,ltv_rank,ltv_dense_rank
0,68,Sara Allison,21137.19,1,1
1,64,Shane Henderson,20066.90,2,2
2,5,Anthony Gonzalez,19595.48,3,3
3,197,Nicholas Johnson,19448.03,4,4
4,165,Ryan Johnson,18969.10,5,5
5,153,Katie Boyd,18274.81,6,6
6,155,Joseph Flores,18184.57,7,7
7,138,Karen Chambers,17774.91,8,8
8,175,Debra Harrington,17619.28,9,9
9,191,Jasmine Graham,17598.02,10,10


In [34]:
pd.read_sql("""
    WITH monthly_revenue AS (
        SELECT strftime('%Y-%m', o.order_date) AS order_month,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS monthly_revenue
        FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY order_month
    )
    SELECT order_month, monthly_revenue,
           ROUND(SUM(monthly_revenue) OVER (ORDER BY order_month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total,
           ROUND(AVG(monthly_revenue) OVER (ORDER BY order_month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3_month
    FROM monthly_revenue
    ORDER BY order_month;
""", conn)


,order_month,monthly_revenue,running_total,moving_avg_3_month
0,2024-08,60575.47,60575.47,60575.47
1,2024-09,65129.00,125704.47,62852.24
2,2024-10,58908.19,184612.66,61537.55
3,2024-11,77271.76,261884.42,67102.98
4,2024-12,50434.44,312318.86,62204.80
5,2025-01,56903.78,369222.64,61536.66
6,2025-02,66309.47,435532.11,57882.56
7,2025-03,59981.12,495513.23,61064.79
8,2025-04,66681.99,562195.22,64324.19
9,2025-05,56177.97,618373.19,60947.03


In [35]:
pd.read_sql("""
    WITH monthly_revenue AS (
        SELECT strftime('%Y-%m', o.order_date) AS order_month,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS monthly_revenue
        FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY order_month
    ),
    with_prev AS (
        SELECT order_month, monthly_revenue,
               LAG(monthly_revenue) OVER (ORDER BY order_month) AS prev_month_revenue
        FROM monthly_revenue
    )
    SELECT order_month, monthly_revenue, prev_month_revenue,
           CASE WHEN prev_month_revenue IS NULL OR prev_month_revenue = 0 THEN NULL
                ELSE ROUND((monthly_revenue - prev_month_revenue) * 100.0 / prev_month_revenue, 2)
           END AS growth_rate_pct
    FROM with_prev
    ORDER BY order_month;
""", conn)


,order_month,monthly_revenue,prev_month_revenue,growth_rate_pct
0,2024-08,60575.47,NaN,NaN
1,2024-09,65129.00,60575.47,7.52
2,2024-10,58908.19,65129.00,-9.55
3,2024-11,77271.76,58908.19,31.17
4,2024-12,50434.44,77271.76,-34.73
5,2025-01,56903.78,50434.44,12.83
6,2025-02,66309.47,56903.78,16.53
7,2025-03,59981.12,66309.47,-9.54
8,2025-04,66681.99,59981.12,11.17
9,2025-05,56177.97,66681.99,-15.75


## 7. Step 6 & 7 — Cohorts, Retention & RFM Segmentation

- Cohort = month of a customer's first order
- Retention = how many customers from a cohort were still ordering in later months
- Churned vs. repeat customers
- Frequency segment + spend tier
- Full RFM (Recency, Frequency, Monetary) scoring with `NTILE(4)`


In [36]:
pd.read_sql("""
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_order_date FROM orders GROUP BY customer_id
    )
    SELECT strftime('%Y-%m', first_order_date) AS cohort_month, COUNT(*) AS num_customers
    FROM first_order
    GROUP BY cohort_month
    ORDER BY cohort_month;
""", conn)


,cohort_month,num_customers
0,2024-08,25
1,2024-09,32
2,2024-10,26
3,2024-11,15
4,2024-12,13
5,2025-01,16
6,2025-02,10
7,2025-03,12
8,2025-04,11
9,2025-05,4


In [37]:
pd.read_sql("""
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_order_date FROM orders GROUP BY customer_id
    ),
    cohorts AS (
        SELECT customer_id, strftime('%Y-%m', first_order_date) AS cohort_month FROM first_order
    ),
    activity AS (
        SELECT DISTINCT customer_id, strftime('%Y-%m', order_date) AS activity_month FROM orders
    )
    SELECT co.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id) AS active_customers
    FROM cohorts co JOIN activity a ON a.customer_id = co.customer_id
    WHERE a.activity_month >= co.cohort_month
    GROUP BY co.cohort_month, a.activity_month
    ORDER BY co.cohort_month, a.activity_month
    LIMIT 20;
""", conn)


,cohort_month,activity_month,active_customers
0,2024-08,2024-08,25
1,2024-08,2024-09,2
2,2024-08,2024-10,4
3,2024-08,2024-11,5
4,2024-08,2024-12,4
5,2024-08,2025-01,6
6,2024-08,2025-02,2
7,2024-08,2025-03,2
8,2024-08,2025-04,5
9,2024-08,2025-05,4


In [38]:
pd.read_sql("""
    WITH order_counts AS (
        SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id
    )
    SELECT CASE WHEN n_orders > 1 THEN 'repeat' ELSE 'one_time_or_churned' END AS customer_type,
           COUNT(*) AS num_customers
    FROM order_counts
    GROUP BY customer_type;
""", conn)


,customer_type,num_customers
0,one_time_or_churned,10
1,repeat,187


In [39]:
pd.read_sql("""
    WITH customer_stats AS (
        SELECT c.customer_id, COUNT(DISTINCT o.order_id) AS n_orders,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spend
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY c.customer_id
    )
    SELECT customer_id, n_orders, total_spend,
           CASE WHEN n_orders = 1 THEN 'one_time'
                WHEN n_orders BETWEEN 2 AND 4 THEN 'occasional'
                ELSE 'loyal' END AS frequency_segment,
           CASE WHEN total_spend < 500 THEN 'low'
                WHEN total_spend BETWEEN 500 AND 2000 THEN 'medium'
                ELSE 'high' END AS spend_tier
    FROM customer_stats
    ORDER BY total_spend DESC
    LIMIT 15;
""", conn)


,customer_id,n_orders,total_spend,frequency_segment,spend_tier
0,68,8,21137.19,loyal,high
1,64,10,20066.90,loyal,high
2,5,9,19595.48,loyal,high
3,197,8,19448.03,loyal,high
4,165,8,18969.10,loyal,high
5,153,7,18274.81,loyal,high
6,155,7,18184.57,loyal,high
7,138,4,17774.91,occasional,high
8,175,9,17619.28,loyal,high
9,191,8,17598.02,loyal,high


In [40]:
pd.read_sql("""
    WITH customer_rfm AS (
        SELECT c.customer_id,
               JULIANDAY((SELECT MAX(order_date) FROM orders)) - JULIANDAY(MAX(o.order_date)) AS recency_days,
               COUNT(DISTINCT o.order_id) AS frequency,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS monetary
        FROM customers c
        JOIN orders o ON o.customer_id = c.customer_id
        JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY c.customer_id
    )
    SELECT customer_id, CAST(recency_days AS INT) AS recency_days, frequency, monetary,
           NTILE(4) OVER (ORDER BY recency_days ASC) AS r_score,
           NTILE(4) OVER (ORDER BY frequency DESC)   AS f_score,
           NTILE(4) OVER (ORDER BY monetary DESC)    AS m_score
    FROM customer_rfm
    ORDER BY monetary DESC
    LIMIT 15;
""", conn)


,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score
0,68,55,8,21137.19,2,1,1
1,64,111,10,20066.90,2,1,1
2,5,136,9,19595.48,3,1,1
3,197,256,8,19448.03,4,1,1
4,165,70,8,18969.10,2,1,1
5,153,154,7,18274.81,3,1,1
6,155,262,7,18184.57,4,1,1
7,138,324,4,17774.91,4,2,1
8,175,235,9,17619.28,4,1,1
9,191,5,8,17598.02,1,1,1


## 8. Step 8 — Simple "CLI-style" reporting tool

In a plain terminal we'd use `argparse` (see `report_cli.py` in the
GitHub version of this project). In a notebook it's more natural to call
a Python function directly — so `run_report()` below works the same way,
just called from a cell instead of the command line.

```python
run_report("top_products", limit=5)
run_report("revenue_by_month")
run_report("rfm", limit=10, save_to="rfm_report.txt")
```


In [41]:
REPORTS = {
    "revenue_by_customer": """
        SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_name,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
        FROM customers c JOIN orders o ON o.customer_id = c.customer_id
        JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY c.customer_id, customer_name ORDER BY total_revenue DESC LIMIT {limit};
    """,
    "revenue_by_category": """
        SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
        FROM order_items oi JOIN products p ON p.product_id = oi.product_id
        GROUP BY p.category ORDER BY total_revenue DESC;
    """,
    "revenue_by_month": """
        SELECT strftime('%Y-%m', o.order_date) AS order_month,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
        FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
        GROUP BY order_month ORDER BY order_month;
    """,
    "top_products": """
        SELECT p.product_id, p.product_name, SUM(oi.quantity) AS total_quantity_sold,
               ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
        FROM order_items oi JOIN products p ON p.product_id = oi.product_id
        GROUP BY p.product_id, p.product_name ORDER BY total_revenue DESC LIMIT {limit};
    """,
    "rank_customers": """
        WITH customer_ltv AS (
            SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_name,
                   ROUND(SUM(oi.quantity * oi.unit_price), 2) AS lifetime_value
            FROM customers c JOIN orders o ON o.customer_id = c.customer_id
            JOIN order_items oi ON oi.order_id = o.order_id
            GROUP BY c.customer_id, customer_name
        )
        SELECT customer_id, customer_name, lifetime_value,
               RANK() OVER (ORDER BY lifetime_value DESC) AS ltv_rank
        FROM customer_ltv ORDER BY lifetime_value DESC LIMIT {limit};
    """,
    "churn_vs_repeat": """
        WITH order_counts AS (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id)
        SELECT CASE WHEN n_orders > 1 THEN 'repeat' ELSE 'one_time_or_churned' END AS customer_type,
               COUNT(*) AS num_customers
        FROM order_counts GROUP BY customer_type;
    """,
    "rfm": """
        WITH customer_rfm AS (
            SELECT c.customer_id,
                   JULIANDAY((SELECT MAX(order_date) FROM orders)) - JULIANDAY(MAX(o.order_date)) AS recency_days,
                   COUNT(DISTINCT o.order_id) AS frequency,
                   ROUND(SUM(oi.quantity * oi.unit_price), 2) AS monetary
            FROM customers c JOIN orders o ON o.customer_id = c.customer_id
            JOIN order_items oi ON oi.order_id = o.order_id
            GROUP BY c.customer_id
        )
        SELECT customer_id, CAST(recency_days AS INT) AS recency_days, frequency, monetary,
               NTILE(4) OVER (ORDER BY recency_days ASC) AS r_score,
               NTILE(4) OVER (ORDER BY frequency DESC) AS f_score,
               NTILE(4) OVER (ORDER BY monetary DESC) AS m_score
        FROM customer_rfm ORDER BY monetary DESC LIMIT {limit};
    """,
}


def run_report(report_name, limit=10, db_conn=None, save_to=None):
    """Notebook-friendly version of the CLI tool's --report option.

    Handles the same edge cases as the terminal CLI:
    empty results, bad limit, unknown report name, DB errors.
    """
    db_conn = db_conn or conn

    if report_name not in REPORTS:
        print(f"Error: unknown report '{report_name}'. Choose from: {sorted(REPORTS.keys())}")
        return None

    if limit is not None and limit <= 0:
        print(f"Error: limit must be positive, got {limit}.")
        return None

    query = REPORTS[report_name].format(limit=limit)

    try:
        df = pd.read_sql(query, db_conn)
    except Exception as e:
        print(f"Database error while running '{report_name}': {e}")
        return None

    if df.empty:
        print(f"No data found for report '{report_name}'. (Empty result set.)")
        return df

    print(f"Report: {report_name}\n" + "=" * (8 + len(report_name)))
    table_str = tabulate(df.values, headers=df.columns, tablefmt="grid")
    print(table_str)

    if save_to:
        with open(save_to, "w") as f:
            f.write(table_str)
        print(f"Saved to: {save_to}")

    return df


# Example calls
run_report("top_products", limit=5)


Report: top_products
+--------------+--------------------+-----------------------+-----------------+
|   product_id | product_name       |   total_quantity_sold |   total_revenue |
+==============+====================+=======================+=================+
|           45 | Actually Painting  |                   164 |         42350   |
+--------------+--------------------+-----------------------+-----------------+
|           44 | Board Turn         |                   134 |         41812.5 |
+--------------+--------------------+-----------------------+-----------------+
|            2 | Service Investment |                   153 |         41327.2 |
+--------------+--------------------+-----------------------+-----------------+
|           17 | Oil Give           |                   125 |         36366.2 |
+--------------+--------------------+-----------------------+-----------------+
|           10 | Do Interview       |                   128 |         36343.6 |
+--------------+---

,product_id,product_name,total_quantity_sold,total_revenue
0,45,Actually Painting,164,42350.03
1,44,Board Turn,134,41812.48
2,2,Service Investment,153,41327.20
3,17,Oil Give,125,36366.18
4,10,Do Interview,128,36343.64


In [42]:
run_report("revenue_by_month")


Report: revenue_by_month
+---------------+-----------------+
| order_month   |   total_revenue |
+===============+=================+
| 2024-08       |        60575.5  |
+---------------+-----------------+
| 2024-09       |        65129    |
+---------------+-----------------+
| 2024-10       |        58908.2  |
+---------------+-----------------+
| 2024-11       |        77271.8  |
+---------------+-----------------+
| 2024-12       |        50434.4  |
+---------------+-----------------+
| 2025-01       |        56903.8  |
+---------------+-----------------+
| 2025-02       |        66309.5  |
+---------------+-----------------+
| 2025-03       |        59981.1  |
+---------------+-----------------+
| 2025-04       |        66682    |
+---------------+-----------------+
| 2025-05       |        56178    |
+---------------+-----------------+
| 2025-06       |        91258    |
+---------------+-----------------+
| 2025-07       |        62202.9  |
+---------------+-----------------+
| 2

,order_month,total_revenue
0,2024-08,60575.47
1,2024-09,65129.00
2,2024-10,58908.19
3,2024-11,77271.76
4,2024-12,50434.44
5,2025-01,56903.78
6,2025-02,66309.47
7,2025-03,59981.12
8,2025-04,66681.99
9,2025-05,56177.97


In [43]:
run_report("rfm", limit=10)


Report: rfm
+---------------+----------------+-------------+------------+-----------+-----------+-----------+
|   customer_id |   recency_days |   frequency |   monetary |   r_score |   f_score |   m_score |
+===============+================+=============+============+===========+===========+===========+
|            68 |             55 |           8 |    21137.2 |         2 |         1 |         1 |
+---------------+----------------+-------------+------------+-----------+-----------+-----------+
|            64 |            111 |          10 |    20066.9 |         2 |         1 |         1 |
+---------------+----------------+-------------+------------+-----------+-----------+-----------+
|             5 |            136 |           9 |    19595.5 |         3 |         1 |         1 |
+---------------+----------------+-------------+------------+-----------+-----------+-----------+
|           197 |            256 |           8 |    19448   |         4 |         1 |         1 |
+-------

,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score
0,68,55,8,21137.19,2,1,1
1,64,111,10,20066.90,2,1,1
2,5,136,9,19595.48,3,1,1
3,197,256,8,19448.03,4,1,1
4,165,70,8,18969.10,2,1,1
5,153,154,7,18274.81,3,1,1
6,155,262,7,18184.57,4,1,1
7,138,324,4,17774.91,4,2,1
8,175,235,9,17619.28,4,1,1
9,191,5,8,17598.02,1,1,1


## 9. Step 9 — Edge case tests

Quick checks that things fail gracefully instead of crashing:
- unknown report name
- non-positive limit
- an empty database (zero orders / zero rows)


In [44]:
# Unknown report name
run_report("does_not_exist")

# Non-positive limit
run_report("top_products", limit=-5)


Error: unknown report 'does_not_exist'. Choose from: ['churn_vs_repeat', 'rank_customers', 'revenue_by_category', 'revenue_by_customer', 'revenue_by_month', 'rfm', 'top_products']
Error: limit must be positive, got -5.


In [45]:
# Empty database test: schema only, no rows at all
empty_conn = sqlite3.connect(":memory:")
empty_conn.executescript(SCHEMA_SQL)
run_report("revenue_by_month", db_conn=empty_conn)
empty_conn.close()


No data found for report 'revenue_by_month'. (Empty result set.)
